<a href="https://colab.research.google.com/github/Swastikagh/Stylia--NeuralStyle-Studio/blob/main/Stylia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install tensorflow gradio pillow matplotlib

import tensorflow as tf
import numpy as np
import gradio as gr
from tensorflow.keras.applications import vgg19
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image

def load_and_process_img(img, target_size=(400, 400)):
    img = img.resize(target_size)
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    return vgg19.preprocess_input(img)

def deprocess_img(x):
    x = x.reshape((x.shape[1], x.shape[2], 3))
    x[:, :, 0] += 103.939
    x[:, :, 1] += 116.779
    x[:, :, 2] += 123.68
    x = x[:, :, ::-1]
    x = np.clip(x, 0, 255).astype('uint8')
    return Image.fromarray(x)

content_layers = ['block5_conv2']
style_layers = [
    'block1_conv1', 'block2_conv1',
    'block3_conv1', 'block4_conv1', 'block5_conv1'
]

def get_model():
    vgg = vgg19.VGG19(include_top=False, weights='imagenet')
    vgg.trainable = False
    style_outputs = [vgg.get_layer(name).output for name in style_layers]
    content_outputs = [vgg.get_layer(name).output for name in content_layers]
    model_outputs = style_outputs + content_outputs
    return tf.keras.models.Model(vgg.input, model_outputs)

def gram_matrix(input_tensor):
    result = tf.linalg.einsum('bijc,bijd->bcd', input_tensor, input_tensor)
    input_shape = tf.shape(input_tensor)
    num_locations = tf.cast(input_shape[1]*input_shape[2], tf.float32)
    return result / num_locations

# =========================================================
# STEP 5️⃣: Style transfer core function
# =========================================================
def run_style_transfer(content_image, style_image,
                       num_iterations=100, content_weight=1e3, style_weight=1e-2):

    model = get_model()
    for layer in model.layers:
        layer.trainable = False

    content_features = model(content_image)
    style_features = model(style_image)

    style_features = [gram_matrix(style_layer) for style_layer in style_features[:len(style_layers)]]
    content_features = [content_layer for content_layer in content_features[len(style_layers):]]

    init_image = tf.Variable(content_image, dtype=tf.float32)
    opt = tf.optimizers.Adam(learning_rate=5.0)

    best_loss, best_img = float('inf'), None

    for i in range(num_iterations):
        with tf.GradientTape() as tape:
            outputs = model(init_image)
            style_output_features = outputs[:len(style_layers)]
            content_output_features = outputs[len(style_layers):]

            style_score = 0
            content_score = 0


            for target_style, comb_style in zip(style_features, style_output_features):
                gram_comb_style = gram_matrix(comb_style)
                style_score += tf.reduce_mean(tf.square(gram_comb_style - target_style))

            for target_content, comb_content in zip(content_features, content_output_features):
                content_score += tf.reduce_mean(tf.square(comb_content - target_content))

            style_score *= style_weight / len(style_layers)
            content_score *= content_weight / len(content_layers)
            loss = style_score + content_score

        grad = tape.gradient(loss, init_image)
        opt.apply_gradients([(grad, init_image)])
        clipped = tf.clip_by_value(init_image, -128, 128)
        init_image.assign(clipped)

        if loss < best_loss:
            best_loss = loss
            best_img = init_image.numpy()

        if i % 20 == 0:
            print(f"Iteration {i}/{num_iterations} - Loss: {loss:.4f}")

    return deprocess_img(best_img)

def style_transfer(content_img, style_img):
    content_array = load_and_process_img(content_img)
    style_array = load_and_process_img(style_img)
    result = run_style_transfer(content_array, style_array, num_iterations=100)
    return result

demo = gr.Interface(
    fn=style_transfer,
    inputs=[
        gr.Image(label="Content Image", type="pil"),
        gr.Image(label="Style Image", type="pil")
    ],
    outputs=gr.Image(label="Stylized Output"),
    title="🎨 Neural Style Transfer",
    description="Upload a content image and a style image to generate a stylized artwork."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1ef35a2302ad9d2dbf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
